In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Les fonctions Qiskit sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.

## Vue d'ensemble
En chimie quantique, le problème de la structure électronique consiste à trouver les solutions à l'équation de Schrödinger électronique — les fonctions d'onde quantiques décrivant le comportement des électrons du système. Ces fonctions d'onde sont des vecteurs d'amplitudes complexes, chaque amplitude correspondant à la contribution d'une configuration électronique possible.

L'état fondamental est la fonction d'onde de plus basse énergie du système et revêt une importance particulière dans l'étude des systèmes moléculaires. L'approche la plus précise pour calculer l'état fondamental prend en compte toutes les configurations électroniques possibles, mais cela devient insoluble pour les systèmes plus grands, car le nombre de configurations croît de façon exponentielle avec la taille du système.

Le Handover Iterative Variational Quantum Eigensolver (HI-VQE) est une méthode hybride quantique-classique innovante pour estimer avec précision l'état fondamental de systèmes moléculaires. Il intègre le matériel quantique et le calcul classique : les processeurs quantiques explorent efficacement les configurations électroniques candidates, tandis que la fonction d'onde résultante est calculée sur des ordinateurs classiques. En générant des fonctions d'onde compactes mais chimiquement précises, HI-VQE favorise la recherche et la découverte en chimie quantique et en science des matériaux.

![Image montrant une vue d'ensemble de l'algorithme HI-VQE de Qunova.](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE réduit la complexité calculatoire du problème de structure électronique en estimant l'état fondamental avec une grande précision. Il se concentre sur un sous-ensemble soigneusement sélectionné des configurations électroniques les plus pertinentes, optimisant à la fois la précision et l'efficacité.

En combinant les atouts des ordinateurs classiques et quantiques, HI-VQE affine et améliore itérativement l'estimation courante de la fonction d'onde. Ses techniques uniques de construction de sous-espace rendent la sélection des configurations plus efficace, offrant aux utilisateurs un meilleur contrôle calculatoire et une précision accrue dans les simulations de chimie quantique.

Pour en savoir plus sur l'algorithme, tu peux [lire l'article de recherche associé](https://arxiv.org/abs/2503.06292).

## Description
Le nombre de configurations électroniques d'un système moléculaire croît de façon exponentielle avec la taille du système. Cependant, pour certains états électroniques, comme l'état fondamental, seule une petite fraction des configurations contribue de manière significative à l'énergie de l'état. Les méthodes d'interaction de configuration sélectionnée (SCI) exploitent cette parcimonie pour réduire les coûts calculatoires en identifiant et en se concentrant sur les configurations les plus pertinentes. Ce sous-ensemble de configurations est appelé sous-espace.

HI-VQE tire parti de l'efficacité inhérente des ordinateurs quantiques pour représenter les systèmes moléculaires et faciliter la recherche de sous-espace. Il intègre des sous-routines classiques et quantiques pour résoudre le problème de structure électronique avec une grande précision. Contrairement aux méthodes SCI quantiques existantes, HI-VQE combine entraînement variationnel, construction itérative de sous-espace et filtrage de configurations par pré-diagonalisation pour améliorer l'efficacité en réduisant le nombre de mesures quantiques, d'itérations et les coûts de diagonalisation classique. HI-VQE peut donc être appliqué à des systèmes moléculaires plus grands nécessitant plus de qubits, et réduit le coût de résolution d'un problème d'une taille donnée au même degré de précision.

![Image montrant une description détaillée de chaque étape de l'algorithme HI-VQE de Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Pour calculer l'état fondamental d'un système, HI-VQE utilise d'abord le package de chimie classique PySCF pour générer une représentation moléculaire à partir des entrées fournies par l'utilisateur, telles que la géométrie moléculaire et d'autres informations moléculaires. Il entre ensuite dans une boucle d'optimisation hybride quantique-classique, affinant itérativement un sous-espace pour représenter optimalement l'état fondamental tout en minimisant le nombre de configurations incluses. La boucle se poursuit jusqu'à ce que les critères de convergence — comme la taille du sous-espace ou la stabilité de l'énergie — soient satisfaits, après quoi la fonction d'onde de l'état fondamental calculée et l'énergie sont produites en sortie. Ces résultats peuvent être utilisés pour construire des surfaces d'énergie potentielle précises et effectuer une analyse plus approfondie du système.

La boucle d'optimisation se concentre sur l'ajustement des paramètres d'un Circuit quantique pour générer un sous-espace de haute qualité. HI-VQE propose trois options de Circuit quantique : [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) et [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). L'optimisation est initialisée près de l'état de référence Hartree-Fock en raison de son adéquation générale. Le Circuit est ensuite exécuté sur un dispositif quantique et des configurations sont échantillonnées à partir de l'état quantique résultant avant d'être renvoyées sous forme de chaînes binaires. En raison du bruit des dispositifs quantiques, certaines configurations échantillonnées peuvent être physiquement invalides, ne conservant pas le nombre d'électrons ou le spin. HI-VQE résout ce problème grâce au processus de récupération de configuration du package [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), permettant aux utilisateurs de corriger les configurations invalides ou de les rejeter.

Les configurations valides passent ensuite par une étape de filtrage optionnelle pour éliminer celles dont la contribution est prédite comme minimale. Cela réduit la dimension du sous-espace, diminuant ainsi le coût de l'étape de diagonalisation. Si le filtrage est activé, un Hamiltonien de sous-espace préliminaire est construit à partir des configurations valides et une diagonalisation est effectuée avec des critères d'arrêt très souples. Bien que la précision des amplitudes résultantes pour chaque configuration soit faible, cela est efficace pour prédire quelles configurations exclure du sous-espace lors de cette itération, et ce calcul est rapide.

Les configurations sélectionnées sont ajoutées au sous-espace, et le Hamiltonien du système est projeté dans ce sous-espace. Le sous-espace se met à jour itérativement, conservant les configurations les plus pertinentes d'une itération à l'autre. Cette approche contraste avec les méthodes alternatives car le Circuit quantique n'a pas besoin d'approximer l'intégralité de l'état fondamental à chaque étape.

Ensuite, le Hamiltonien de sous-espace est diagonalisé classiquement pour obtenir la valeur propre la plus basse et son vecteur propre correspondant, représentant une approximation de l'état fondamental et de son énergie. À mesure que la qualité du sous-espace s'améliore au fil des itérations, l'état fondamental calculé se rapproche davantage du vrai état fondamental. Une étape de filtrage supplémentaire peut être effectuée à ce stade pour retirer du sous-espace toute configuration ne contribuant pas substantiellement à l'état fondamental calculé. Cette étape garantit que le sous-espace transmis à l'itération suivante est aussi compact que possible. Elle est évaluée sur la base des amplitudes renvoyées par la diagonalisation, qui représentent la contribution d'importance de chaque configuration à l'état fondamental calculé.

Un test de convergence détermine ensuite si un entraînement supplémentaire améliorerait les résultats. Dans l'affirmative, une étape d'expansion classique optionnelle est effectuée, les paramètres du Circuit quantique sont mis à jour pour minimiser davantage l'énergie calculée, et le processus recommence. L'étape d'expansion classique génère des configurations supplémentaires pour le sous-espace, venant compléter les configurations échantillonnées depuis le dispositif quantique. Elle identifie d'abord la configuration ayant la plus grande amplitude dans les résultats de diagonalisation, puis génère de nouvelles configurations avec des excitations simples et doubles à partir de la configuration identifiée. Le nombre souhaité de ces configurations est ensuite ajouté au sous-espace.

Une fois qu'il est déterminé que les itérations ont convergé, HI-VQE renvoie l'état fondamental calculé (sous la forme des états dans le sous-espace et leurs amplitudes dans la fonction d'onde de l'état fondamental), son énergie, et une mesure de la variance d'énergie indiquant si l'état calculé forme un état propre du Hamiltonien du système.

Les utilisateurs peuvent choisir le Circuit quantique utilisé et le nombre de shots pris pour chaque Circuit quantique, ainsi que contrôler la taille du sous-espace ou activer la génération classique de configurations supplémentaires pour compléter les configurations générées quantiquement. Ainsi, les utilisateurs peuvent adapter le comportement de HI-VQE à leurs applications.

## Démarrage
D'abord, [demande l'accès à la fonction](https://forms.office.com/r/zN3hvMTqJ1).
Ensuite, authentifie-toi avec ta [clé API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) et, en supposant que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local, sélectionne la fonction Qiskit comme suit :

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Entrées
Consulte le tableau suivant pour tous les paramètres d'entrée acceptés par la fonction. Les sections [Options de molécule](#molecule-options) et [Options HI-VQE](#hi-vqe-options) ci-dessous entrent plus en détail sur ces arguments.

| Nom                    | Type                                                           | Description                                                                                                                                                                                                                                                                                                 | Requis | Défaut                                                                   | Exemple                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Peut être une chaîne de caractères ou des listes structurées contenant des paires atome-coordonnées. Si fourni sous forme de chaîne, la géométrie moléculaire doit être en format de coordonnées cartésiennes. Si fourni sous forme de liste, il doit s'agir d'une liste de listes contenant chacune une chaîne d'atome et un tuple de coordonnées. | Oui    | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` ou `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nom du Backend sur lequel effectuer la requête.                                                                                                                                                                                                                                                             | Oui    | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | La dimension maximale du sous-espace pour la diagonalisation. Moins d'états seront utilisés si le nombre n'est pas un carré parfait.                                                                                                                                                                        | Oui    | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Le nombre maximum d'états CI générés classiquement à inclure à chaque itération.                                                                                                                                                                                                                            | Oui    | N/A                                                                      | `10`                                                                                      |
| molecule_options       | dict                                                           | Options relatives à la molécule utilisée en entrée de HI-VQE. Voir la section [Options de molécule](#molecule-options) pour plus de détails.                                                                                                                                                                | Non    | Voir la section [Options de molécule](#molecule-options) pour les détails. | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options          | dict                                                           | Options contrôlant le comportement de l'algorithme HI-VQE. Voir la section [Options HI-VQE](#hi-vqe-options) pour plus de détails.                                                                                                                                                                         | Non    | Voir la section [Options HI-VQE](#hi-vqe-options) pour les détails.      | `{"shots": 10_000, "max_iter": 10 }`                               |

### Options de molécule
Le tableau suivant détaille toutes les clés et valeurs pouvant être définies dans le dictionnaire `molecule_options`, ainsi que leurs types de données et valeurs par défaut. Toutes les clés sont optionnelles.

| Clé                               | Type de valeur                      | Valeur par défaut                | Plage valide                                                                                                                                                   | Explication                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Diverses                                                                                                                                                       | Un entier spécifiant la charge nette totale du système moléculaire. La valeur par défaut est 0 ; elle peut cependant être n'importe quel entier.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Diverses                                                                                                                                                       | Une chaîne spécifiant le type de base ; ces valeurs sont passées à pyscf. (ex. : `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                          |
| `"active_orbitals"`               | `List[int]`                         | Tous les indices d'orbitales.    | Les indices d'orbitales spatiales valides pour le problème.                                                                                                    | Une liste d'indices d'orbitales actives dans l'intervalle [0, n) où n est le nombre de qubits utilisés dans le problème. Si ceci est spécifié, l'argument frozen_orbitals doit également être spécifié.                                                                                                                                                                                                                                                                                                                                  |
| `"frozen_orbitals"`               | `List[int]`                         | Aucun indice.                    | Les indices d'orbitales spatiales valides pour le problème, à l'exclusion des orbitales actives.                                                               | Une liste d'indices d'orbitales gelées dans la même plage que active_orbitals. Si spécifié, active_orbitals doit également être spécifié. Note que seules les orbitales occupées doivent être gelées, car le nombre d'électrons actifs est réduit de 2 pour chaque orbitale occupée gelée.                                                                                                                                                                                                                                               |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbitales moléculaires Hartree-Fock. | Diverses.                                                                                                                                                  | Les coefficients des orbitales spatiales utilisés dans le calcul des intégrales de répulsion électronique du système. Quelques exemples valides sont les orbitales moléculaires Hartree-Fock, les orbitales naturelles et les orbitales AVAS.                                                                                                                                                                                                                                                                                            |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` ou `False`                                                                                                                                              | Utilisé pour invoquer la symétrie de groupe ponctuel pour les calculs moléculaires initiaux afin de construire la base orbitale adaptée à la symétrie. Ces orbitales adaptées à la symétrie sont utilisées comme fonctions de base pour les calculs SCF suivants. La valeur par défaut est False ; si défini à True, la symétrie sera invoquée et les groupes ponctuels arbitraires seront automatiquement détectés et utilisés. Si une symétrie particulière est assignée, par exemple symmetry = "Dooh", une erreur sera levée si la géométrie moléculaire n'est pas soumise à cette symétrie requise. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Peut être utilisé pour générer un sous-groupe de la symétrie détectée. Cela n'a aucun effet lorsque la symétrie est spécifiée via le paramètre symmetry.                                                                                                                                                                                                                                                                                                                                                                                 |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Spécifie l'unité de mesure à utiliser pour les coordonnées atomiques et les distances. La valeur par défaut est l'angström.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Spécifie le modèle nucléaire à utiliser. Par défaut, le modèle nucléaire ponctuel est utilisé ; d'autres valeurs activent le modèle nucléaire gaussien. Si une fonction est fournie, elle sera utilisée avec le modèle nucléaire gaussien pour générer la valeur de distribution de charge nucléaire « zeta ».                                                                                                                                                                                                                            |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Spécifie le pseudopotentiel pour les atomes de la molécule. La valeur par défaut est None, indiquant qu'aucun pseudopotentiel n'est appliqué et que tous les électrons sont explicitement inclus dans les calculs.                                                                                                                                                                                                                                                                                                                        |
| `"cart"`                          | `bool`                              | `False`                          | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Spécifie si les GTOs cartésiennes doivent être utilisées comme fonctions de base du moment angulaire dans le calcul. La valeur par défaut False utilisera les GTOs sphériques.                                                                                                                                                                                                                                                                                                                                                           |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Voir la [documentation pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                  | Définit le moment magnétique de spin colinéaire de chaque atome. Par défaut, cette valeur est None et chaque atome est initialisé avec un spin de zéro.                                                                                                                                                                                                                                                                                                                                                                                  |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | ex. ["H 1s", "O 2p"] pour H2O                                                                                                                                 | Définit les orbitales atomiques à inclure dans le schéma AVAS. Voir la [documentation AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                           |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Entre 0.0 et 2.0                                                                                                                                               | Spécifie la valeur de coupure utilisée pour déterminer quelles orbitales atomiques (AO) sont conservées dans l'espace actif.                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` ou `"ccsd"`                                                                                                                                            | Définit l'approche théorique pour préparer les orbitales naturelles et sélectionner les orbitales actives selon le schéma des nombres d'occupation d'orbitales naturelles (NOONs). Voir la [documentation NOONs](https://doi.org/10.1063/5.0042147). Les indices d'orbitales actives et gelées doivent tous deux être fournis pour contrôler le nombre d'orbitales (et le nombre de qubits).                                                                                                                                              |

### Options HI-VQE
Le tableau suivant détaille toutes les clés et valeurs pouvant être définies dans le dictionnaire `hivqe_options`, ainsi que leurs types de données et valeurs par défaut. Toutes les clés sont optionnelles.

| Clé                               | Type de valeur                      | Valeur par défaut                | Plage valide                                                                                                                                                   | Explication                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Entre 1 et 10 000.                                                                                                                                             | Nombre de shots à utiliser sur le dispositif quantique par itération.                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"max_iter"`                      | `int`                               | `25`                             | Entre 1 et 50.                                                                                                                                                 | Le nombre maximum d'itérations à exécuter pour optimiser l'ansatz. L'algorithme peut utiliser moins d'itérations si la convergence est atteinte plus tôt.                                                                                                                                                                                                                                                                                                                                                                               |
| `"initial_basis_states"`          | `List[str]`                         | L'état Hartree-Fock.             | Chaînes binaires dont le nombre de bits correspond au nombre de qubits requis pour le problème.                                                                | Peut être utilisé pour redémarrer l'algorithme avec des états classiques issus d'un résultat précédent.                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` ou `"lucj"`                                                                                                                                   | Spécifie l'ansatz quantique à optimiser pour générer de nouveaux états. `"epa"` sélectionne l'ansatz préservant les excitations. `"hea"` sélectionne l'ansatz efficace matériellement. `"lucj"` sélectionne l'ansatz local unitary cluster Jastrow.                                                                                                                                                                                                                                                                                      |
| `"convergence_count"`             | `int`                               | `3`                              | Au moins 2.                                                                                                                                                    | Le nombre d'itérations sans changement significatif de l'énergie calculée devant s'écouler avant que l'algorithme soit considéré comme ayant convergé.                                                                                                                                                                                                                                                                                                                                                                                  |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Supérieur à 0 et au plus 1.                                                                                                                                    | L'amplitude de changement de l'énergie calculée considérée comme significative pour les vérifications de convergence.                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` ou `False`                                                                                                                                              | Si `True`, les itérations `convergence_count` doivent se produire sans changement significatif interrupteur pour être qualifiées de convergentes. Si `False`, l'algorithme s'arrêtera après `convergence_count` si des changements insignifiants se sont produits lors de n'importe quelle itération pendant le processus d'optimisation.                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` ou `False`.                                                                                                                                             | Si la récupération de configuration du package `qiskit-addon-sqd` doit être utilisée ou non. Si True, les états invalides échantillonnés depuis le dispositif quantique sont corrigés classiquement. Si False, ils sont rejetés.                                                                                                                                                                                                                                                                                                         |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | L'une des valeurs suivantes : `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` ou `"sca"`. Si l'ansatz `"lucj"` est utilisé, `"lucj_default"` est aussi une option. | Spécifie le schéma d'intrication à utiliser dans le Circuit quantique, suivant les conventions Qiskit et [ffsim pour l'ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Supérieur à 0.                                                                                                                                                 | Le nombre de répétitions de chaque couche dans le Circuit quantique.                                                                                                                                                                                                                                                                                                                                                                                                                                                                    |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Au moins 0 et inférieur à 1.                                                                                                                                   | La tolérance pour décider quels états doivent être filtrés hors du sous-espace après diagonalisation. Elle spécifie le seuil d'inclusion pour les états du sous-espace en fonction de leurs amplitudes calculées.                                                                                                                                                                                                                                                                                                                        |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Entre `1e-4` et `1e-1`, inclus.                                                                                                                                | La tolérance pour prédire quels états doivent être filtrés hors du sous-espace avant la diagonalisation. Elle contrôle la précision des amplitudes prédites pour chaque état, une valeur plus petite donnant des prédictions plus précises.                                                                                                                                                                                                                                                                                              |

## Sorties
La fonction renvoie un dictionnaire avec quatre clés et valeurs. Les clés et valeurs sont documentées dans le tableau suivant :

| Clé             | Type de valeur | Explication                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | L'énergie approximative de l'état fondamental de la molécule.                                                             |
| `"states"`      | `List[str]`   | Les déterminants sélectionnés formant le sous-espace utilisé pour résoudre l'énergie. Ils sont en format alpha-bêta alternant. |
| `"eigenvector"` | `List[float]` | Le vecteur propre correspondant à l'état fondamental du sous-espace composé des `"states"`.                               |
| `"energy_variance"` | `float` | La variance d'énergie de l'état fondamental du sous-espace composé des `"states"`, donnant une indication de la qualité de la solution. Cette valeur est non négative et une valeur plus faible signifie que l'état fondamental du sous-espace se rapproche davantage d'un état propre du Hamiltonien du système. |
| `"energy_history"` | `List[float]` | Les énergies calculées à chaque itération pendant le processus d'optimisation hybride, dans l'ordre où elles ont été calculées. Deux énergies sont calculées par itération dans le cadre du processus d'optimisation SPSA. |

## Exemple
Le premier exemple montre comment calculer l'énergie de l'état fondamental d'une molécule NH3 à l'aide de l'algorithme HI-VQE.

#### Définir la géométrie moléculaire et les options
La géométrie moléculaire de NH3 est fournie avec des coordonnées cartésiennes séparées par « ; » pour chaque atome.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Des options supplémentaires peuvent être définies et fournies pour le système moléculaire dans le format de dictionnaire suivant.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Exécute la fonction avec les entrées de géométrie et d'options.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Il est conseillé d'afficher l'ID du job de la fonction afin de pouvoir le fournir dans les demandes de support en cas de problème.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Cet exemple utilise ensuite 16 qubits avec 8 orbitales de base sto3g pour une molécule NH3.
Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail de fonction Qiskit ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Une fois le job terminé, les résultats peuvent être obtenus avec l'instance `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Pour accéder à l'énergie de l'état fondamental, utilise la clé `"energy"`. La clé `"eigenvector"` fournit les coefficients CI avec la notation en chaîne de bits correspondante de la configuration électronique stockée dans `"states"` des résultats.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Résultat :

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936

## Performance
Cette section présente les calculs de référence démontrés de HI-VQE avec un cas à 24 qubits pour Li2S, un cas à 40 qubits pour une molécule N2 et un cas à 44 qubits pour un système FeP-NO.

#### Courbe de surface d'énergie potentielle de dissociation pour une molécule Li2S avec 24 qubits
La courbe PES est montrée avec la référence FCI et l'estimation initiale de RHF, ainsi que l'erreur d'énergie par rapport à la référence FCI.

![Image montrant que HI-VQE produit des solutions dans la précision chimique d'une courbe PES de référence classique pour le système Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Les calculs ont été effectués avec les géométries et options suivantes.